# Feature Engineering — SPY / EUR-USD / Gold

Builds the 5-min execution-layer feature set for all three markets, with 1H and Daily context features merged in without lookahead.

**Scope:** features only, no labels yet (triple-barrier labeling is a separate next step, once we've eyeballed these).

**Design notes:**
- **No lookahead**: every higher-timeframe (1H/Daily) feature is only made available to a 5-min row once that higher-timeframe bar has *fully closed* (`available_at` = bar timestamp + its duration), joined via `merge_asof(..., direction="backward")`.
- **No forward-filling of missing 5-min bars**: SPY's `5min_rth.csv` has NaN placeholder rows for the ~0.5% of gaps found during RTH cleaning — those get dropped here rather than faked as flat candles.
- **Volume caveat**: EUR/USD and Gold from Twelve Data are OTC/spot instruments with no centralized volume, so volume-derived features (volume ratio, VWAP) are SPY-only. Don't expect those columns for the other two markets.
- **Warmup rows are dropped, not filled**: slow indicators (EMA-50, ADX, ATR) need history to stabilize — the first ~60 rows of each series get trimmed rather than imputed.

Install if needed: `pip install ta --break-system-packages`

## 0. Config

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from ta.trend import EMAIndicator, MACD, ADXIndicator
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.volatility import AverageTrueRange, BollingerBands

DATA_DIR = Path("data")

# has_volume: only SPY has real (exchange) volume; Twelve Data forex/gold don't.
# session_based: SPY has a fixed 9:30-16:00 ET session; forex/gold trade ~24h, so
# session-relative features (opening range, minutes-since-open, VWAP) don't apply there.
MARKETS = {
    "spy":    {"5min": "5min_rth.csv", "1h": "1h.csv", "daily": "daily.csv", "has_volume": True,  "session_based": True},
    "eurusd": {"5min": "5min.csv",     "1h": "1h.csv", "daily": "daily.csv", "has_volume": False, "session_based": False},
    "gold":   {"5min": "5min.csv",     "1h": "1h.csv", "daily": "daily.csv", "has_volume": False, "session_based": False},
}

FEATURE_WARMUP_ROWS = 60  # trimmed after computing indicators, see section 4

## 1. Load bars

In [ ]:
def load_bars(market: str, interval: str) -> pd.DataFrame:
    cfg = MARKETS[market]
    path = DATA_DIR / market / cfg[interval]
    df = pd.read_csv(path)

    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    elif "datetime_et" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime_et"], utc=True)
    else:
        raise ValueError(f"No datetime column found in {path}")

    df = df.sort_values("datetime").reset_index(drop=True)
    if "close" in df.columns:
        df = df.dropna(subset=["close"])  # drops SPY's RTH placeholder gap-rows
    return df

## 2. Execution-layer features (5-min)

Trend, momentum, volatility, support/resistance, breakout, and session/time features computed directly on the 5-min bars.

In [ ]:
def compute_execution_features(df: pd.DataFrame, has_volume: bool, session_based: bool) -> pd.DataFrame:
    df = df.copy().set_index("datetime")

    # --- trend ---
    for span in [9, 21, 50]:
        ema = EMAIndicator(df["close"], window=span).ema_indicator()
        df[f"ema_{span}"] = ema
        df[f"dist_ema_{span}_pct"] = (df["close"] - ema) / ema * 100

    macd = MACD(df["close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()
    df["macd_hist"] = macd.macd_diff()

    df["adx"] = ADXIndicator(df["high"], df["low"], df["close"], window=14).adx()

    # --- momentum ---
    df["rsi_14"] = RSIIndicator(df["close"], window=14).rsi()
    stoch = StochasticOscillator(df["high"], df["low"], df["close"], window=14, smooth_window=3)
    df["stoch_k"] = stoch.stoch()
    df["stoch_d"] = stoch.stoch_signal()
    df["roc_10"] = ROCIndicator(df["close"], window=10).roc()

    # --- volatility ---
    atr = AverageTrueRange(df["high"], df["low"], df["close"], window=14)
    df["atr_14"] = atr.average_true_range()
    df["atr_pct"] = df["atr_14"] / df["close"] * 100
    bb = BollingerBands(df["close"], window=20, window_dev=2)
    df["bb_width_pct"] = (bb.bollinger_hband() - bb.bollinger_lband()) / df["close"] * 100
    df["bb_pct_b"] = bb.bollinger_pband()
    df["realized_vol_20"] = np.log(df["close"] / df["close"].shift(1)).rolling(20).std()

    if has_volume and "volume" in df.columns:
        df["volume_ratio_20"] = df["volume"] / df["volume"].rolling(20).mean()

    # --- support/resistance & breakout ---
    lookback = 20
    prior_high = df["high"].shift(1).rolling(lookback).max()
    prior_low = df["low"].shift(1).rolling(lookback).min()
    df[f"dist_to_{lookback}bar_high_pct"] = (prior_high - df["close"]) / df["close"] * 100
    df[f"dist_to_{lookback}bar_low_pct"] = (df["close"] - prior_low) / df["close"] * 100
    df[f"breakout_up_{lookback}"] = (df["close"] > prior_high).astype(int)
    df[f"breakout_down_{lookback}"] = (df["close"] < prior_low).astype(int)

    # --- session / time ---
    et_index = df.index.tz_convert("America/New_York")
    minutes_of_day = et_index.hour * 60 + et_index.minute
    df["tod_sin"] = np.sin(2 * np.pi * minutes_of_day / (24 * 60))
    df["tod_cos"] = np.cos(2 * np.pi * minutes_of_day / (24 * 60))
    df["dow"] = et_index.dayofweek

    if session_based:
        df["session_date"] = et_index.date
        df["minutes_since_open"] = minutes_of_day - (9 * 60 + 30)
        df["minutes_to_close"] = (16 * 60) - minutes_of_day

        or_mask = df["minutes_since_open"] < 30
        or_levels = df[or_mask].groupby("session_date").agg(or_high=("high", "max"), or_low=("low", "min"))
        df = df.join(or_levels, on="session_date")
        df["or_breakout_up"] = (df["close"] > df["or_high"]).astype(int)
        df["or_breakout_down"] = (df["close"] < df["or_low"]).astype(int)

        if has_volume and "volume" in df.columns:
            typical_price = (df["high"] + df["low"] + df["close"]) / 3
            pv = typical_price * df["volume"]
            cum_pv = pv.groupby(df["session_date"]).cumsum()
            cum_vol = df["volume"].groupby(df["session_date"]).cumsum()
            df["vwap"] = cum_pv / cum_vol
            df["dist_vwap_pct"] = (df["close"] - df["vwap"]) / df["vwap"] * 100

        df = df.drop(columns=["session_date"])

    return df.reset_index()

## 3. Higher-timeframe context features (1H, Daily)

Computed on the higher-timeframe bars themselves, tagged with `available_at` = the timestamp at which that bar is fully closed (and therefore safe to use). Includes daily pivot points, which only really make sense off the Daily frame but are computed generically so the function works for both.

In [ ]:
def compute_context_features(df: pd.DataFrame, timeframe_hours: float) -> pd.DataFrame:
    df = df.copy().set_index("datetime")

    ema20 = EMAIndicator(df["close"], window=20).ema_indicator()
    df["ctx_ema20_slope"] = ema20.diff()
    df["ctx_trend_up"] = (df["close"] > ema20).astype(int)

    df["ctx_adx"] = ADXIndicator(df["high"], df["low"], df["close"], window=14).adx()

    atr_pct = AverageTrueRange(df["high"], df["low"], df["close"], window=14).average_true_range() / df["close"] * 100
    df["ctx_atr_pct"] = atr_pct
    vol_rank = atr_pct.rolling(60, min_periods=20).rank(pct=True)
    df["ctx_vol_regime"] = pd.cut(vol_rank, bins=[0, 0.33, 0.66, 1.0], labels=["low", "normal", "high"])

    df["ctx_pivot"] = (df["high"] + df["low"] + df["close"]) / 3
    df["ctx_r1"] = 2 * df["ctx_pivot"] - df["low"]
    df["ctx_s1"] = 2 * df["ctx_pivot"] - df["high"]

    # bar only becomes usable once fully closed -- this is what merge_context() guards against lookahead with
    df["available_at"] = df.index + pd.Timedelta(hours=timeframe_hours)

    keep_cols = [c for c in df.columns if c.startswith("ctx_")] + ["available_at"]
    return df[keep_cols].reset_index(drop=True)

## 4. Merge context onto the 5-min execution rows

In [ ]:
def merge_context(base: pd.DataFrame, context: pd.DataFrame, prefix: str) -> pd.DataFrame:
    base = base.sort_values("datetime").reset_index(drop=True)
    context = context.sort_values("available_at").reset_index(drop=True)
    context = context.rename(columns={c: f"{prefix}_{c}" for c in context.columns if c != "available_at"})
    merged = pd.merge_asof(base, context, left_on="datetime", right_on="available_at", direction="backward")
    return merged.drop(columns=["available_at"])

## 5. Run the pipeline for all three markets

In [ ]:
all_features = {}

for market, cfg in MARKETS.items():
    print(f"=== {market} ===")
    df_5min = load_bars(market, "5min")
    df_1h = load_bars(market, "1h")
    df_daily = load_bars(market, "daily")

    exec_feat = compute_execution_features(df_5min, has_volume=cfg["has_volume"], session_based=cfg["session_based"])
    ctx_1h = compute_context_features(df_1h, timeframe_hours=1)
    ctx_daily = compute_context_features(df_daily, timeframe_hours=24)

    merged = merge_context(exec_feat, ctx_1h, prefix="h1")
    merged = merge_context(merged, ctx_daily, prefix="d1")

    before = len(merged)
    merged = merged.iloc[FEATURE_WARMUP_ROWS:].reset_index(drop=True)
    # some rows still won't have 1H/Daily context yet at the very start of history -- drop those too
    merged = merged.dropna(subset=["ema_50", "atr_14", "h1_ctx_adx", "d1_ctx_adx"]).reset_index(drop=True)
    print(f"  {before:,} -> {len(merged):,} rows after warmup/NaN trim")

    out_path = DATA_DIR / market / "features_5min.csv"
    merged.to_csv(out_path, index=False)
    print(f"  saved -> {out_path}  ({merged.shape[1]} columns)")

    all_features[market] = merged

## 6. Quick QC

In [ ]:
for market, df in all_features.items():
    print(f"--- {market} ---")
    print(f"shape: {df.shape}")
    remaining_nans = df.isna().sum()
    remaining_nans = remaining_nans[remaining_nans > 0]
    if len(remaining_nans):
        print("columns still with NaNs:")
        print(remaining_nans)
    else:
        print("no remaining NaNs")
    print()

In [ ]:
# eyeball a sample of columns for one market
all_features["spy"].filter(regex="^(datetime|close|adx|rsi_14|atr_pct|h1_ctx_trend_up|d1_ctx_vol_regime)$").head(10)

---

**Next steps, not in this notebook:**
- Correlation check across the feature set (several of these — e.g. RSI and Stochastic — are naturally correlated; worth a heatmap before modeling, though tree models tolerate correlated features fine, so this is a diagnostic, not a mandatory pruning step).
- Triple-barrier labeling (ATR-based SL/TP barriers + max holding time) — separate notebook, built on top of `features_5min.csv`.
- Feature selection proper happens *after* a first model pass (XGBoost feature importance), not before — better to over-include here and let the model tell you what matters.